In [1]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


In [3]:
DATA_DIR = Path("/content/drive/MyDrive/CSCI 5527 Project/Dataset Preprocessing/processed_eeg_dataset/processed_eeg_dataset")

X_eeg = np.load(DATA_DIR / "X_eeg.npy", mmap_mode="r")
metadata = pd.read_csv(DATA_DIR / "eeg_metadata_with_sss.csv")
y_cls = np.load(DATA_DIR / "y_labels.npy", allow_pickle=True)
groups = np.load(DATA_DIR / "groups.npy", allow_pickle=True)

print("X_eeg shape :", X_eeg.shape)
print("metadata    :", metadata.shape)
print("y_cls       :", y_cls.shape)
print("groups      :", groups.shape)
print(metadata.columns.tolist())

X_eeg shape : (8300, 61, 1250)
metadata    : (8300, 14)
y_cls       : (8300,)
groups      : (8300,)
['subject_id', 'session', 'task', 'label', 'label_name', 'epoch_index', 'sfreq', 'n_channels', 'n_times', 'img_channels', 'img_height', 'img_width', 'file_path', 'SSS']


In [4]:
assert len(X_eeg) == len(metadata) == len(y_cls) == len(groups)
assert np.array_equal(metadata["label"].to_numpy(), y_cls)
assert np.array_equal(metadata["subject_id"].astype(str).to_numpy(), groups.astype(str))

print("Alignment check passed.")

Alignment check passed.


# Preparing Data

In [5]:
metadata["SSS"] = pd.to_numeric(metadata["SSS"], errors="coerce")

sss_unique = sorted(metadata["SSS"].dropna().unique().tolist())
sss_to_rank = {v: i for i, v in enumerate(sss_unique)}
rank_to_sss = {i: v for v, i in sss_to_rank.items()}

print("Observed SSS values:", sss_unique)
print("Number of ordinal levels:", len(sss_unique))
print("Mapping:", sss_to_rank)

Observed SSS values: [1.0, 2.0, 3.0, 4.0, 5.0, 7.0]
Number of ordinal levels: 6
Mapping: {1.0: 0, 2.0: 1, 3.0: 2, 4.0: 3, 5.0: 4, 7.0: 5}


In [6]:
metadata["sss_rank"] = metadata["SSS"].map(sss_to_rank)

print(metadata[["SSS", "sss_rank"]].drop_duplicates().sort_values("SSS"))
print("Missing SSS rows:", metadata["SSS"].isna().sum())

      SSS  sss_rank
60    1.0       0.0
300   2.0       1.0
0     3.0       2.0
600   4.0       3.0
780   5.0       4.0
2040  7.0       5.0
120   NaN       NaN
Missing SSS rows: 3920


# Preparing Data

In [7]:
TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15

all_indices = np.arange(len(y_cls))

gss_1 = GroupShuffleSplit(n_splits=1, train_size=TRAIN_SIZE, random_state=SEED)
train_idx, temp_idx = next(gss_1.split(all_indices, y_cls, groups))

temp_y = y_cls[temp_idx]
temp_groups = groups[temp_idx]
relative_val = VAL_SIZE / (VAL_SIZE + TEST_SIZE)

gss_2 = GroupShuffleSplit(n_splits=1, train_size=relative_val, random_state=SEED + 1)
temp_val_idx, temp_test_idx = next(gss_2.split(np.arange(len(temp_idx)), temp_y, temp_groups))

val_idx = temp_idx[temp_val_idx]
test_idx = temp_idx[temp_test_idx]

print("train:", len(train_idx))
print("val  :", len(val_idx))
print("test :", len(test_idx))

train: 5738
val  : 1246
test : 1316


In [8]:
train_groups = set(groups[train_idx].astype(str))
val_groups = set(groups[val_idx].astype(str))
test_groups = set(groups[test_idx].astype(str))

assert len(train_groups & val_groups) == 0
assert len(train_groups & test_groups) == 0
assert len(val_groups & test_groups) == 0

print("No subject leakage.")

No subject leakage.


In [9]:
def ordinal_encode(rank, num_classes):
    """
    rank: integer in [0, num_classes-1]
    returns tensor of shape (num_classes - 1,)
    """
    levels = torch.zeros(num_classes - 1, dtype=torch.float32)
    levels[:rank] = 1.0
    return levels

In [10]:
class EEGMultiTaskDataset(Dataset):
    def __init__(self, x_eeg, metadata_df, indices, num_sss_classes):
        self.x_eeg = x_eeg
        self.metadata_df = metadata_df.reset_index(drop=True)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.num_sss_classes = num_sss_classes

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        global_idx = int(self.indices[idx])

        x = np.array(self.x_eeg[global_idx], dtype=np.float32, copy=True)
        x = torch.from_numpy(x).unsqueeze(0)   # (1, 61, 1250)

        row = self.metadata_df.iloc[global_idx]

        y_cls = torch.tensor(int(row["label"]), dtype=torch.long)

        if pd.notna(row["sss_rank"]):
            sss_rank = int(row["sss_rank"])
            sss_levels = ordinal_encode(sss_rank, self.num_sss_classes)
            sss_mask = torch.tensor(1.0, dtype=torch.float32)
        else:
            sss_rank = -1
            sss_levels = torch.zeros(self.num_sss_classes - 1, dtype=torch.float32)
            sss_mask = torch.tensor(0.0, dtype=torch.float32)

        meta = {
            "global_index": global_idx,
            "subject_id": row["subject_id"],
            "session": row["session"],
            "file_path": row["file_path"],
            "epoch_index": int(row["epoch_index"]),
            "SSS": row["SSS"] if pd.notna(row["SSS"]) else np.nan,
        }

        return x, y_cls, sss_levels, sss_mask, meta

In [11]:
num_sss_classes = len(sss_unique)

train_dataset = EEGMultiTaskDataset(X_eeg, metadata, train_idx, num_sss_classes)
val_dataset = EEGMultiTaskDataset(X_eeg, metadata, val_idx, num_sss_classes)
test_dataset = EEGMultiTaskDataset(X_eeg, metadata, test_idx, num_sss_classes)

train_labels = metadata.iloc[train_idx]["label"].to_numpy(dtype=np.int64)
class_counts = np.bincount(train_labels, minlength=2)
sample_weights = 1.0 / np.maximum(class_counts[train_labels], 1)

train_sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Train class counts:", class_counts)

Train class counts: [2940 2798]


In [14]:
class ConvBNAct(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, act="relu"):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)

        if act == "relu":
            self.act = nn.ReLU(inplace=True)
        elif act == "leaky_relu":
            self.act = nn.LeakyReLU(0.1, inplace=True)
        elif act == "silu":
            self.act = nn.SiLU(inplace=True)
        else:
            raise ValueError(f"Unsupported activation: {act}")

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

In [15]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        pooled = self.pool(x).view(b, c)
        scale = self.fc(pooled).view(b, c, 1, 1)
        return x * scale

In [16]:
class ResidualBlock(nn.Module):
    def __init__(self, channels, dropout=0.10):
        super().__init__()
        self.block = nn.Sequential(
            ConvBNAct(channels, channels, kernel_size=3, padding=1, act="silu"),
            nn.Dropout2d(dropout),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        return self.act(x + self.block(x))

In [17]:
def initialize_custom_model(module):
    for m in module.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, (nn.BatchNorm2d, nn.LayerNorm)):
            if getattr(m, "weight", None) is not None:
                nn.init.ones_(m.weight)
            if getattr(m, "bias", None) is not None:
                nn.init.zeros_(m.bias)

In [18]:
class MultiTaskResidualEEGCNN(nn.Module):
    def __init__(self, num_cls=2, num_sss_classes=7):
        super().__init__()

        self.stem = nn.Sequential(
            ConvBNAct(1, 32, kernel_size=(7, 31), padding=(3, 15), act="leaky_relu"),
            nn.MaxPool2d(kernel_size=(2, 4)),
            ConvBNAct(32, 64, kernel_size=(5, 15), padding=(2, 7), act="silu"),
            nn.MaxPool2d(kernel_size=(2, 4)),
        )

        self.res_stack = nn.Sequential(
            ResidualBlock(64, dropout=0.10),
            SEBlock(64),
            ConvBNAct(64, 128, kernel_size=3, padding=1, act="silu"),
            nn.MaxPool2d(kernel_size=(2, 2)),
            ResidualBlock(128, dropout=0.15),
            SEBlock(128),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.feature_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.LayerNorm(256),
            nn.SiLU(inplace=True),
            nn.Dropout(0.55),
        )

        self.cls_head = nn.Linear(256, num_cls)
        self.sss_head = nn.Linear(256, num_sss_classes - 1)

        initialize_custom_model(self)

    def forward(self, x):
        x = self.stem(x)
        x = self.res_stack(x)
        feats = self.feature_head(x)

        cls_logits = self.cls_head(feats)
        sss_logits = self.sss_head(feats)

        return cls_logits, sss_logits, feats

In [19]:
model = MultiTaskResidualEEGCNN(num_cls=2, num_sss_classes=num_sss_classes).to(DEVICE)
print(model)

MultiTaskResidualEEGCNN(
  (stem): Sequential(
    (0): ConvBNAct(
      (conv): Conv2d(1, 32, kernel_size=(7, 31), stride=(1, 1), padding=(3, 15), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): LeakyReLU(negative_slope=0.1, inplace=True)
    )
    (1): MaxPool2d(kernel_size=(2, 4), stride=(2, 4), padding=0, dilation=1, ceil_mode=False)
    (2): ConvBNAct(
      (conv): Conv2d(32, 64, kernel_size=(5, 15), stride=(1, 1), padding=(2, 7), bias=False)
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (3): MaxPool2d(kernel_size=(2, 4), stride=(2, 4), padding=0, dilation=1, ceil_mode=False)
  )
  (res_stack): Sequential(
    (0): ResidualBlock(
      (block): Sequential(
        (0): ConvBNAct(
          (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=1e-05, momentum=

In [20]:
cls_weights = len(train_labels) / (2 * np.maximum(class_counts, 1))
cls_weights = torch.tensor(cls_weights, dtype=torch.float32, device=DEVICE)

cls_criterion = nn.CrossEntropyLoss(weight=cls_weights)
ord_criterion = nn.BCEWithLogitsLoss(reduction="none")

In [21]:
def multitask_loss(cls_logits, y_cls, sss_logits, sss_levels, sss_mask, lambda_sss=1.0):
    # NS/SD classification loss
    cls_loss = cls_criterion(cls_logits, y_cls)

    # Ordinal SSS loss only where label exists
    ord_loss_raw = ord_criterion(sss_logits, sss_levels)  # (B, K-1)
    ord_loss_per_sample = ord_loss_raw.mean(dim=1)        # (B,)

    if sss_mask.sum() > 0:
        ord_loss = (ord_loss_per_sample * sss_mask).sum() / sss_mask.sum()
    else:
        ord_loss = torch.tensor(0.0, device=cls_logits.device)

    total = cls_loss + lambda_sss * ord_loss
    return total, cls_loss, ord_loss

In [22]:
def decode_ordinal_logits(sss_logits):
    probs = torch.sigmoid(sss_logits)
    pred_rank = (probs > 0.5).sum(dim=1)
    return pred_rank

In [23]:
def classification_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, average="binary"),
    }

In [28]:
def regression_like_metrics_from_ordinal(true_vals, pred_vals):
    true_vals = np.asarray(true_vals, dtype=np.float32)
    pred_vals = np.asarray(pred_vals, dtype=np.float32)

    mask = np.isfinite(true_vals) & np.isfinite(pred_vals)
    true_vals = true_vals[mask]
    pred_vals = pred_vals[mask]

    if len(true_vals) == 0:
        return {
            "MAE": np.nan,
            "RMSE": np.nan,
            "Pearson": np.nan,
            "Spearman": np.nan,
        }

    return {
        "MAE": mean_absolute_error(true_vals, pred_vals),
        "RMSE": np.sqrt(mean_squared_error(true_vals, pred_vals)),
        "Pearson": pearsonr(true_vals, pred_vals)[0] if len(np.unique(true_vals)) > 1 else np.nan,
        "Spearman": spearmanr(true_vals, pred_vals)[0] if len(np.unique(true_vals)) > 1 else np.nan,
    }

In [29]:
def evaluate_multitask(model, loader, device):
    model.eval()

    all_cls_true, all_cls_pred = [], []
    all_sss_true, all_sss_pred = [], []

    running_loss = 0.0
    running_cls_loss = 0.0
    running_ord_loss = 0.0
    n_batches = 0

    with torch.no_grad():
        for x, y_cls, sss_levels, sss_mask, meta in loader:
            x = x.to(device)
            y_cls = y_cls.to(device)
            sss_levels = sss_levels.to(device)
            sss_mask = sss_mask.to(device)

            cls_logits, sss_logits, _ = model(x)
            loss, cls_loss, ord_loss = multitask_loss(
                cls_logits, y_cls, sss_logits, sss_levels, sss_mask, lambda_sss=1.0
            )

            running_loss += loss.item()
            running_cls_loss += cls_loss.item()
            running_ord_loss += ord_loss.item()
            n_batches += 1

            cls_pred = torch.argmax(cls_logits, dim=1)
            all_cls_true.extend(y_cls.cpu().numpy())
            all_cls_pred.extend(cls_pred.cpu().numpy())

            pred_rank = decode_ordinal_logits(sss_logits).cpu().numpy()

            sss_values = meta["SSS"]
            for i in range(len(pred_rank)):
                true_sss = float(sss_values[i])

                if np.isfinite(true_sss):
                    pred_sss = float(rank_to_sss[int(pred_rank[i])])
                    all_sss_true.append(true_sss)
                    all_sss_pred.append(pred_sss)

    out = {
        "loss": running_loss / max(n_batches, 1),
        "cls_loss": running_cls_loss / max(n_batches, 1),
        "ord_loss": running_ord_loss / max(n_batches, 1),
        "cls_metrics": classification_metrics(np.array(all_cls_true), np.array(all_cls_pred)),
    }

    if len(all_sss_true) > 0:
        out["sss_metrics"] = regression_like_metrics_from_ordinal(
            np.array(all_sss_true), np.array(all_sss_pred)
        )
    else:
        out["sss_metrics"] = None

    return out

In [30]:
optimizer = AdamW(model.parameters(), lr=2e-4, weight_decay=2e-3)

NUM_EPOCHS = 35
PATIENCE = 8
LAMBDA_SSS = 1.0

In [31]:
best_val_metric = -np.inf
best_state = None
stale_epochs = 0
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    running_cls_loss = 0.0
    running_ord_loss = 0.0

    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}", leave=False)

    for x, y_cls, sss_levels, sss_mask, meta in progress:
        x = x.to(DEVICE)
        y_cls = y_cls.to(DEVICE)
        sss_levels = sss_levels.to(DEVICE)
        sss_mask = sss_mask.to(DEVICE)

        optimizer.zero_grad()

        cls_logits, sss_logits, _ = model(x)
        loss, cls_loss, ord_loss = multitask_loss(
            cls_logits, y_cls, sss_logits, sss_levels, sss_mask, lambda_sss=LAMBDA_SSS
        )

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        running_cls_loss += cls_loss.item() * x.size(0)
        running_ord_loss += ord_loss.item() * x.size(0)

    train_loss = running_loss / len(train_loader.dataset)
    train_cls_loss = running_cls_loss / len(train_loader.dataset)
    train_ord_loss = running_ord_loss / len(train_loader.dataset)

    val_out = evaluate_multitask(model, val_loader, DEVICE)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_cls_loss": train_cls_loss,
        "train_ord_loss": train_ord_loss,
        "val_loss": val_out["loss"],
        "val_cls_loss": val_out["cls_loss"],
        "val_ord_loss": val_out["ord_loss"],
        "val_cls_acc": val_out["cls_metrics"]["accuracy"],
        "val_cls_bal_acc": val_out["cls_metrics"]["balanced_accuracy"],
        "val_cls_f1": val_out["cls_metrics"]["f1"],
    }

    if val_out["sss_metrics"] is not None:
        row.update({
            "val_sss_mae": val_out["sss_metrics"]["MAE"],
            "val_sss_rmse": val_out["sss_metrics"]["RMSE"],
            "val_sss_pearson": val_out["sss_metrics"]["Pearson"],
            "val_sss_spearman": val_out["sss_metrics"]["Spearman"],
        })

    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_cls_bal_acc={val_out['cls_metrics']['balanced_accuracy']:.4f} | "
        f"val_cls_f1={val_out['cls_metrics']['f1']:.4f} | "
        f"val_sss_rmse={val_out['sss_metrics']['RMSE']:.4f} | "
        f"val_sss_pearson={val_out['sss_metrics']['Pearson']:.4f}"
    )

    # pick best model using combination of tasks
    current_metric = (
        val_out["cls_metrics"]["balanced_accuracy"]
        + 0.1 * val_out["sss_metrics"]["Pearson"]
    )

    if current_metric > best_val_metric:
        best_val_metric = current_metric
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        stale_epochs = 0
    else:
        stale_epochs += 1
        if stale_epochs >= PATIENCE:
            print("Early stopping triggered.")
            break

Epoch 1/35:   0%|          | 0/180 [00:00<?, ?it/s]

Epoch 01 | train_loss=0.9844 | val_cls_bal_acc=0.5908 | val_cls_f1=0.6671 | val_sss_rmse=1.6477 | val_sss_pearson=0.0773


Epoch 2/35:   0%|          | 0/180 [00:00<?, ?it/s]

Epoch 02 | train_loss=0.8837 | val_cls_bal_acc=0.7454 | val_cls_f1=0.7002 | val_sss_rmse=1.4876 | val_sss_pearson=0.0076


Epoch 3/35:   0%|          | 0/180 [00:00<?, ?it/s]

...

Epoch 11 | train_loss=0.3367 | val_cls_bal_acc=0.7100 | val_cls_f1=0.6769 | val_sss_rmse=1.6794 | val_sss_pearson=0.0496


Epoch 12/35:   0%|          | 0/180 [00:00<?, ?it/s]

Epoch 12 | train_loss=0.3146 | val_cls_bal_acc=0.5713 | val_cls_f1=0.6601 | val_sss_rmse=1.5688 | val_sss_pearson=0.1273


Epoch 13/35:   0%|          | 0/180 [00:00<?, ?it/s]

Epoch 13 | train_loss=0.2658 | val_cls_bal_acc=0.5701 | val_cls_f1=0.6655 | val_sss_rmse=1.2612 | val_sss_pearson=0.0413
Early stopping triggered.
